# 08 — Agent Document Workflow (Flagship)

End-to-end walkthrough of the FactPy **agent layer** on a real document, real
SQLite ledger, and **real OpenAI LLM extraction**. This notebook covers every
stage of Layer 4C:

- **4C1** — Document staging (format detection, segmentation, provenance)
- **4C3a** — Single-segment LLM extraction
- **4C3b** — Batch extraction over all segments of a document
- **4C3c** — Entity resolution (dedupe / merge)
- **4C2** — Draft bundle creation, review, and commit to ledger
- Ledger inspection via `ReadReviewOrchestrator.query_claims` and friends

Unlike notebooks 06 / 07 (which mock their adapter runners), **this notebook is
end-to-end real**: real SQLite ledger via `open_runtime_session`, real
`LocalRuntimeAPI`, real `ReadReviewOrchestrator`, real `commit_bundle`, and
real OpenAI LLM extraction through `ExtractionAgent`.

**Prerequisites:**
- `OPENAI_API_KEY` environment variable must be set.
- Python packages `instructor` and `litellm` must be installed (required by the
  default LLM client inside `ExtractionAgent`).
- Extraction variance at `temperature=0.0` is expected per OBS-01
  (B3 iter 4 addendum). Re-running this notebook may produce different valid /
  proposal counts on the same input; the `committed_count > 0` guarantee is
  stable, the exact number is not.

**Next:** none — this is the final notebook in the public learning path.

**Related blueprints (all archived):** See §12 at the bottom for the full chain
of agent layer blueprints this notebook exercises.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

## 0. Imports + pre-flight checks

The notebook requires a live OpenAI API key and the optional LLM dependency
stack. We fail loudly up-front so we never reach the extraction cell with a
half-configured environment.

In [2]:
from __future__ import annotations

import os
import shutil
import tempfile
from pathlib import Path
from pprint import pprint

# ---------------------------------------------------------------------------
# Pre-flight 1: verify the Python environment has all dependencies BEFORE we
# start importing kernel. kernel core depends on `pydantic>=2`
# and `diskcache>=5`; the agent extraction layer additionally requires
# `instructor>=1.6` and `litellm>=1.50` for real LLM calls. If any are
# missing we fail here with a clear install hint instead of letting a
# deep-link ImportError surface from inside kernel.
# ---------------------------------------------------------------------------
_missing: list[str] = []
for _mod, _pkg_spec in (
    ("pydantic", "pydantic>=2"),
    ("diskcache", "diskcache>=5"),
    ("instructor", "instructor>=1.6"),
    ("litellm", "litellm>=1.50"),
):
    try:
        __import__(_mod)
    except ImportError:
        _missing.append(_pkg_spec)

if _missing:
    raise RuntimeError(
        "Notebook 08 requires the following Python packages which are not\n"
        f"installed in the current Jupyter kernel environment: {_missing}\n\n"
        "The recommended fix is to install kernel itself in editable\n"
        "mode with its `extraction` optional dependencies. From the repo root:\n\n"
        "    pip install -e '.[extraction]'\n\n"
        "If the notebook is already running under a dedicated kernel (e.g. a\n"
        "conda env or a venv), activate that environment first and then run\n"
        "the pip command there, NOT in your default shell. After installing,\n"
        "restart the Jupyter kernel and re-run this cell."
    )

# Pre-flight 2: API key must be set for real LLM extraction in cell 15.
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set in the current environment. This notebook\n"
        "performs real LLM calls via litellm and cannot run without a valid\n"
        "key. Set the env var (e.g. `export OPENAI_API_KEY=sk-...`) and\n"
        "restart the Jupyter kernel."
    )

print("Pre-flight OK:")
print("  - pydantic / diskcache / instructor / litellm installed")
print("  - OPENAI_API_KEY is set")

# ---------------------------------------------------------------------------
# Now it's safe to import kernel. Every symbol below is re-exported
# from `agent.__init__` or the nearest public submodule.
# ---------------------------------------------------------------------------
from kernel.sdk import (
    Entity,
    Field,
    Identity,
    compile_schema_from_classes,
)
from agent import (
    AgentCheckpointStore,
    AgentScope,
    AgentSession,
    BundleManager,
    BundleReviewAction,
    CandidatePayloadCache,
    DocumentStaging,
    DraftManager,
    ReadReviewOrchestrator,
    RuntimeBootstrapSpec,
    WriteTools,
)
from agent.extraction import (
    BatchExtractionConfig,
    BatchExtractionError,
    BatchExtractionResult,
    BatchExtractor,
    EntityResolver,
    ExtractionAgent,
    ExtractionConfig,
    ResolutionConfig,
    ResolutionError,
    ResolutionResult,
)
from agent.tools._runtime_api import LocalRuntimeAPI
from agent.tools.evaluate import EvaluateTools
from agent.tools.explain import ExplainTools
from agent.tools.kg_read import KGReadTools
from service.runtime_v1 import (
    close_runtime_session,
    open_runtime_session,
)

print("kernel imports OK.")

Pre-flight OK:
  - pydantic / diskcache / instructor / litellm installed
  - OPENAI_API_KEY is set
factpy_kernel imports OK.


## 1. Define domain entity classes

We model a tiny project-documentation domain: a `Document` entity with a
summary and free-form tags, and a `Module` entity with a description and owner.
Both use natural-string identities (`title` / `name`) so the LLM can propose
them directly from document text without inventing synthetic IDs.

Only field predicates are used in this notebook — no `Relationship` — because
single-segment LLM extraction is most reliable on attribute facts. Adding
relationship predicates is a separate exercise (see
`test_relationship_schema.py` for the DSL pattern).

The `module:description` predicate name deliberately matches the B3 load test
harness (`docs/references/working/load-test-2026-04-11/`) so this notebook can
double as reference material for anyone revisiting the B3 commit path later.

In [3]:
class Document(Entity):
    title: str = Identity(primary_key=True)
    summary: str = Field(
        cardinality="single",
        description="Short description of what the document contains.",
    )
    tag: str = Field(
        cardinality="multi",
        description="Topic tag applied to the document.",
    )


class Module(Entity):
    name: str = Identity(primary_key=True)
    description: str = Field(
        cardinality="single",
        description="What this module does, in one sentence.",
    )
    owner: str = Field(
        cardinality="single",
        description="Team or individual responsible for the module.",
    )


print("Defined entity classes: Document, Module")

Defined entity classes: Document, Module


## 2. Compile canonical SchemaIR

`compile_schema_from_classes` is **canonical-by-construction**: the returned
SchemaIR is guaranteed to satisfy the runtime validator inside
`service.runtime_v1.open_runtime_session`. In particular it produces all three
of the canonical rules that the B3 load test harness had to discover through
smoke tests (canonical top-level keys, non-empty `arg_specs[].name`, and
`group_key_indexes` as a list on every predicate).

We verify those three rules in-place as a smoke check.

In [4]:
schema_ir = compile_schema_from_classes([Document, Module])

pred_ids = [p["pred_id"] for p in schema_ir["predicates"]]
print(f"Compiled {len(schema_ir['entities'])} entities, {len(schema_ir['predicates'])} predicates")
print("Predicates:")
for pred_id in sorted(pred_ids):
    print(f"  - {pred_id}")

# Canonical smoke: the three rules B3 discovered through three smoke attempts.
assert "_note" not in schema_ir, "canonical rule 1: no underscore-prefixed top-level keys"
for pred in schema_ir["predicates"]:
    for arg in pred["arg_specs"]:
        assert arg.get("name"), (
            f"canonical rule 2: every arg_spec must have a non-empty name "
            f"(violated on {pred['pred_id']})"
        )
    assert isinstance(pred.get("group_key_indexes"), list), (
        f"canonical rule 3: group_key_indexes must be a list "
        f"(violated on {pred['pred_id']})"
    )

print("\nCanonical validator pre-check: PASS (rules 1 / 2 / 3 all satisfied)")

Compiled 2 entities, 8 predicates
Predicates:
  - Document:exists
  - Module:exists
  - document:summary
  - document:tag
  - document:title
  - module:description
  - module:name
  - module:owner

Canonical validator pre-check: PASS (rules 1 / 2 / 3 all satisfied)


## 3. Open runtime session + wire AgentSession

We run everything under a temporary directory so the SQLite ledger and the
agent's burr checkpoint database can be cleanly discarded at the end. The
`AgentSession` is bound to the runtime session via `bind_runtime_session`,
which is the single state transition the B3 load test harness was missing
before its commit path blueprint was implemented.

In [5]:
workdir = Path(tempfile.mkdtemp(prefix="notebook08_"))
ledger_path = workdir / "runtime-ledger.db"
burr_db_path = workdir / "agent-burr.sqlite3"
print(f"Working directory: {workdir}")

open_dto: dict[str, object] = {
    "schema_ir": schema_ir,
    "ledger_path": str(ledger_path),
}
open_resp = open_runtime_session(open_dto)
assert open_resp["ok"], open_resp
runtime_session_id = open_resp["session"]["session_id"]
print(f"Opened runtime session: {runtime_session_id}")

scope = AgentScope(
    agent_id="examples-notebook-08",
    allowed_entity_types=frozenset({"Document", "Module"}),
    allowed_pred_ids=frozenset(
        {
            "document:summary",
            "document:tag",
            "module:description",
            "module:owner",
        }
    ),
    require_source=True,
)
agent_session = AgentSession(scope=scope, status="active")
agent_session.bind_runtime_session(
    runtime_session_id,
    bootstrap_spec=RuntimeBootstrapSpec.from_open_dto(open_dto),
    burr_db_path=str(burr_db_path),
)

candidate_cache = CandidatePayloadCache(burr_db_path)
checkpoint_store = AgentCheckpointStore(burr_db_path)
runtime_api = LocalRuntimeAPI()

print(f"Bound agent session: {agent_session.agent_session_id}")

Working directory: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/notebook08_kgwg9vck
Opened runtime session: rt_9fb6b27dddd3435c93c9d7741f1818ba
Bound agent session: agent_a2dd73cb3c95


## 4. Build the `ReadReviewOrchestrator` stack

The orchestrator is the single public facade for every agent action — read,
review, write. It composes five tool families (`KGReadTools`, `ExplainTools`,
`EvaluateTools`, `WriteTools`, `DocumentStaging`) plus the draft / bundle /
cache / checkpoint state. This wiring matches `test_agent_l4c2_workflow.py`
exactly — the only difference is that this notebook drives it with a real LLM.

In [6]:
draft_manager = DraftManager()
bundle_manager = BundleManager(draft_manager=draft_manager)
document_staging = DocumentStaging()

kg_read_tools = KGReadTools(runtime_api=runtime_api)
explain_tools = ExplainTools(runtime_api=runtime_api)
evaluate_tools = EvaluateTools(
    runtime_api=runtime_api,
    candidate_cache=candidate_cache,
    explain_tools=explain_tools,
    session=agent_session,
)
write_tools = WriteTools(runtime_api=runtime_api, session=agent_session)

orchestrator = ReadReviewOrchestrator(
    session=agent_session,
    draft_manager=draft_manager,
    kg_read_tools=kg_read_tools,
    explain_tools=explain_tools,
    evaluate_tools=evaluate_tools,
    candidate_cache=candidate_cache,
    checkpoint_store=checkpoint_store,
    write_tools=write_tools,
    document_staging=document_staging,
    bundle_manager=bundle_manager,
)
print("Orchestrator ready.")

Orchestrator ready.


## 5. Stage an inline document (4C1)

`DocumentStaging.stage_document` handles format detection (txt / md / pdf /
docx) and segmentation. We feed it an inline Markdown string describing a
fictional project so the LLM has a well-structured fact surface to extract.

In [7]:
readme_markdown = """# README.md \u2014 Project Alpha

Project Alpha is a distributed data processing pipeline composed of several
cooperating modules. It is designed for high-throughput ingestion and
low-latency retrieval of resolved entity snapshots.

## Modules

The `ingest` module is owned by the data-infra team. It reads incoming JSON
events from Kafka, validates their schema, and writes them to object storage.
The module sustains roughly fifty thousand events per second at peak load.

The `resolve` module is owned by Alice. It performs entity resolution across
ingested records using a combination of exact and fuzzy matching. Resolution
results are cached in Redis for downstream queries.

The `serve` module is owned by the platform team. It exposes a gRPC API that
returns resolved entity snapshots to external clients with sub-ten-millisecond
latency on cache hits.

All three modules share a common logging and metrics infrastructure built on
OpenTelemetry.
""".encode("utf-8")

staging_result = orchestrator.stage_document(
    doc_name="README.md",
    content=readme_markdown,
)
if hasattr(staging_result, "error_kind"):
    raise RuntimeError(f"Staging failed: {staging_result}")

print(f"doc_id           : {staging_result.source.doc_id}")
print(f"doc_name         : {staging_result.source.doc_name}")
print(f"doc_type         : {staging_result.source.doc_type}")
print(f"parser           : {staging_result.parser_name} v{staging_result.parser_version}")
print(f"byte_size        : {staging_result.source.byte_size}")
print(f"total_chars      : {staging_result.total_chars}")
print(f"segments         : {len(staging_result.segments)}")
print(f"high_clarity     : {staging_result.high_clarity_count}")

for idx, seg in enumerate(staging_result.segments):
    preview = seg.raw_text.replace("\n", " ")[:72]
    print(f"  seg[{idx}] clarity={seg.structural_clarity:.2f} chars={seg.char_offset_start}-{seg.char_offset_end}  {preview!r}")

doc_id           : 997c784df65baf90
doc_name         : README.md
doc_type         : md
parser           : txt v0.1.0
byte_size        : 951
total_chars      : 948
segments         : 7
high_clarity     : 0
  seg[0] clarity=0.20 chars=0-27  '# README.md — Project Alpha'
  seg[1] clarity=0.20 chars=29-226  'Project Alpha is a distributed data processing pipeline composed of seve'
  seg[2] clarity=0.20 chars=228-238  '## Modules'
  seg[3] clarity=0.20 chars=240-468  'The `ingest` module is owned by the data-infra team. It reads incoming J'
  seg[4] clarity=0.20 chars=470-675  'The `resolve` module is owned by Alice. It performs entity resolution ac'
  seg[5] clarity=0.20 chars=677-855  'The `serve` module is owned by the platform team. It exposes a gRPC API '
  seg[6] clarity=0.20 chars=857-948  'All three modules share a common logging and metrics infrastructure buil'


## 6. Real LLM extraction (4C3a + 4C3b)

This is the one cell that makes real OpenAI API calls. We construct an
`ExtractionAgent` with `gpt-4o-mini` at `temperature=0.0` (matching B3 iter 5)
and wrap it in a `BatchExtractor` so every segment is processed in a first
pass, with low-yield segments optionally re-examined in a second gleaning
pass.

**Variance warning (OBS-01)**: running this cell multiple times will produce
slightly different `total_proposals` / `valid_specs` counts even at
`temperature=0.0`. This is a documented observation from B3 iter 4. The
downstream pipeline is robust to the variance — as long as `aggregated_specs`
is non-empty, every subsequent step will succeed.

In [8]:
extraction_agent = ExtractionAgent(
    config=ExtractionConfig(
        model="gpt-4o-mini",
        max_retries=2,
        temperature=0.0,
        max_text_chars=4000,
        timeout_seconds=30.0,
    ),
)
batch_extractor = BatchExtractor(
    extraction_agent=extraction_agent,
    batch_config=BatchExtractionConfig(
        max_segments_per_batch=1000,
        early_stop_on_batch_cap_reached=True,
        enable_gleaning=True,
    ),
)

batch_outcome = batch_extractor.extract_batch(
    segments=list(staging_result.segments),
    schema_ir=schema_ir,
    scope=scope,
)
if isinstance(batch_outcome, BatchExtractionError):
    raise RuntimeError(f"Batch extraction failed: {batch_outcome}")
assert isinstance(batch_outcome, BatchExtractionResult)

m = batch_outcome.metrics
print(f"segments processed    : {m.total_segments}")
print(f"success / error       : {m.success_segment_count} / {m.error_segment_count}")
print(f"total proposals       : {m.total_proposal_count}")
print(f"valid specs           : {m.total_valid_count}")
print(f"rejections            : {m.total_rejection_count}")
print(f"batch duration        : {m.batch_duration_ms} ms")
print(f"aggregated_specs len  : {len(batch_outcome.aggregated_specs)}")

if batch_outcome.aggregated_specs:
    print("\nFirst 5 draft specs (entity_type / pred_id / identity / value):")
    for spec in batch_outcome.aggregated_specs[:5]:
        value_preview = str(spec.field_values[0]) if spec.field_values else "-"
        print(f"  - {spec.entity_type:10s} {spec.pred_id:22s} {spec.entity_identity}  ->  {value_preview}")

segments processed    : 7
success / error       : 7 / 0
total proposals       : 8
valid specs           : 7
rejections            : 1
batch duration        : 12970 ms
aggregated_specs len  : 7

First 5 draft specs (entity_type / pred_id / identity / value):
  - Document   document:summary       {'title': '997c784df65baf90'}  ->  ('string', 'Project Alpha is a distributed data processing pipeline composed of several cooperating modules. It is designed for high-throughput ingestion and low-latency retrieval of resolved entity snapshots.')
  - Module     module:owner           {'name': 'ingest'}  ->  ('string', 'data-infra team')
  - Module     module:description     {'name': 'ingest'}  ->  ('string', 'reads incoming JSON events from Kafka, validates their schema, and writes them to object storage.')
  - Module     module:owner           {'name': 'resolve'}  ->  ('string', 'Alice')
  - Module     module:description     {'name': 'resolve'}  ->  ('string', 'performs entity resolution across

## 7. Entity resolution (4C3c)

The `EntityResolver` dedupes draft specs that refer to the same entity. For
example, if the LLM proposed the `ingest` module twice (once with a description
and once with an owner), resolution merges them into one entity with both
facts attached.

In [9]:
entity_resolver = EntityResolver(
    config=ResolutionConfig(
        enable_dedupe=True,
        max_input_specs=10000,
    ),
)

resolve_outcome = entity_resolver.resolve_batch(list(batch_outcome.aggregated_specs))
if isinstance(resolve_outcome, ResolutionError):
    raise RuntimeError(f"Resolution failed: {resolve_outcome}")
assert isinstance(resolve_outcome, ResolutionResult)

print(f"doc_id               : {resolve_outcome.doc_id}")
print(f"input specs          : {len(batch_outcome.aggregated_specs)}")
print(f"resolved specs       : {len(resolve_outcome.resolved_specs)}")
print(f"merge events         : {len(resolve_outcome.merge_events)}")
print(f"dedupe reduction     : {len(batch_outcome.aggregated_specs) - len(resolve_outcome.resolved_specs)}")

doc_id               : 997c784df65baf90
input specs          : 7
resolved specs       : 7
merge events         : 0
dedupe reduction     : 0


## 8. Create a document bundle (4C2)

`create_document_bundle` is the first of the four orchestrator calls that make
up the commit path. It persists the draft specs as `FactDraft` records keyed by
a `bundle_id` and checkpoints the agent state into the burr database.

In [10]:
if not resolve_outcome.resolved_specs:
    raise RuntimeError(
        "The LLM produced zero valid specs for this document. "
        "This is possible under OBS-01 variance — re-run cell 6 before proceeding."
    )

bundle = orchestrator.create_document_bundle(
    source_document_id=staging_result.source.doc_id,
    source_document_name=staging_result.source.doc_name,
    facts=list(resolve_outcome.resolved_specs),
)

print(f"bundle_id     : {bundle.bundle_id}")
print(f"status        : {bundle.status}")
print(f"draft_count   : {len(bundle.draft_ids)}")
print(f"first 3 drafts: {bundle.draft_ids[:3]}")

bundle_id     : bundle_6b9d367aec0c
status        : created
draft_count   : 7
first 3 drafts: ['draft_056f82e9c9d8', 'draft_b940d93553fd', 'draft_c6bb76493020']


## 9. Review and commit the bundle (4C2)

The remaining three calls of the 4-call commit sequence: `open_bundle_review`
transitions the bundle into a reviewable state, `apply_bundle_review` records
per-draft approve / reject decisions, and `commit_bundle` writes the approved
drafts to the real SQLite ledger. The result structure reports total / committed
/ rejected / failed counts and per-item `WriteResult` details.

For demo purposes we approve every draft. In a real pipeline a human reviewer
(or a higher-confidence acceptance policy) would select a subset.

In [11]:
orchestrator.open_bundle_review(bundle.bundle_id)

review_actions = [
    BundleReviewAction(draft_id=draft_id, action="approve")
    for draft_id in bundle.draft_ids
]
reviewed = orchestrator.apply_bundle_review(bundle.bundle_id, review_actions)
print(f"reviewed status: {reviewed.status}")

commit_result = orchestrator.commit_bundle(
    bundle.bundle_id,
    kind="add",
    confirmed_by="notebook-08-demo",
)
print(f"total          : {commit_result.total}")
print(f"committed_count: {commit_result.committed_count}")
print(f"rejected_count : {commit_result.rejected_count}")
print(f"failed_count   : {commit_result.failed_count}")

assert commit_result.committed_count > 0, (
    "Expected at least one successful commit. This is the stable guarantee "
    "even under OBS-01 variance."
)

reviewed status: approved
total          : 7
committed_count: 7
rejected_count : 0
failed_count   : 0


## 10. Inspect the ledger state

The commit is now durable in the SQLite ledger. We use `KGReadTools.query_claims`
to read back the committed facts for each predicate, and print the full
provenance metadata each claim carries (source document, segment, character
range, approving reviewer, and the agent executor id).

In [12]:
schema_summary = orchestrator.get_schema()
print("Schema summary (from orchestrator.get_schema):")
print(f"  digest       : {schema_summary.get('schema_digest', '?')}")
print(f"  entity_types : {[e['entity_type'] for e in schema_summary.get('entity_types', [])]}")
print(f"  predicates   : {[p['pred_id'] for p in schema_summary.get('predicates', [])]}")

for pred_id in sorted(scope.allowed_pred_ids):
    claims = orchestrator.query_claims(pred_id=pred_id)
    print(f"\n{pred_id}: {len(claims)} claim(s)")
    for claim in claims:
        value = claim.rest_terms[0] if claim.rest_terms else None
        source = claim.meta.get("source", "?")
        approved_by = claim.meta.get("approved_by", "?")
        print(f"  - value={value}  source={source}  approved_by={approved_by}")

print("\nCommitted drafts (from orchestrator.list_committed_drafts):")
for draft in orchestrator.list_committed_drafts():
    print(f"  - draft_id={draft.draft_id}  entity={draft.entity_type}  pred={draft.pred_id}  status={draft.status}")

Schema summary (from orchestrator.get_schema):
  digest       : sha256:cb9897b83616fb92a53d210c7be3ad8edb4c0f1fc896d2abe053d426fbf75994
  entity_types : ['Document', 'Module']
  predicates   : ['Document:exists', 'document:title', 'document:summary', 'document:tag', 'Module:exists', 'module:name', 'module:description', 'module:owner']

document:summary: 1 claim(s)
  - value=('string', 'Project Alpha is a distributed data processing pipeline composed of several cooperating modules. It is designed for high-throughput ingestion and low-latency retrieval of resolved entity snapshots.')  source=doc:README.md:seg:997c784d_0001  approved_by=notebook-08-demo

document:tag: 0 claim(s)

module:description: 3 claim(s)
  - value=('string', 'reads incoming JSON events from Kafka, validates their schema, and writes them to object storage.')  source=doc:README.md:seg:997c784d_0003  approved_by=notebook-08-demo
  - value=('string', 'performs entity resolution across ingested records using a combinatio

## 10.5 Adversarial Identity Fragmentation Diagnostic

**Purpose**: determine whether P2 (cascaded ER / fuzzy identity matching)
has enough evidence to open. We stage an adversarial document where the
same logical entity is referred to by multiple names ("Kafka Ingest
Pipeline" / "ingest" / "the ingestion service") and check whether the
current resolver merges them or leaves them fragmented.

**P2 evidence standard** (3 conditions, ALL required):
1. Same logical entity extracted with ≥2 different identity keys
2. Human-verifiable as the same entity
3. Resolver did NOT merge them

If condition 3 holds, P2 is justified. Otherwise, P2 stays shelved.

In [ ]:
# ── Adversarial document: same entities, multiple name variants ──
adversarial_text = """\
# Internal Architecture Notes

The Kafka Ingest Pipeline is the backbone of our data infrastructure.
It was built by the data-infra team in Q3 2025.

## Performance Characteristics

ingest handles approximately 50,000 events per second at peak load.
The ingestion service validates JSON schemas before writing to S3.

## Entity Resolution

Alice Chen owns the resolve module. She implemented the fuzzy matching
algorithm that powers entity deduplication across the pipeline.

## API Layer

The serve module exposes a gRPC API. As described in the architecture
overview, the pipeline described above feeds data into this API layer.
Alice's team also contributes to serve maintenance.

## Project Summary

Project Alpha (also known simply as Alpha) is the umbrella project
that encompasses all three modules mentioned above.
""".encode("utf-8")

# Stage
adv_staging = orchestrator.stage_document(
    doc_name="architecture_notes.md",
    content=adversarial_text,
)
if hasattr(adv_staging, "error_kind"):
    raise RuntimeError(f"Adversarial staging failed: {adv_staging}")
print(f"Adversarial doc: {len(adv_staging.segments)} segments")
for idx, seg in enumerate(adv_staging.segments):
    preview = seg.raw_text.replace('\n', ' ')[:72]
    print(f"  seg[{idx}] clarity={seg.structural_clarity:.2f}  {preview!r}")

# Extract (with entity context + gleaning)
adv_batch = batch_extractor.extract_batch(
    segments=list(adv_staging.segments),
    schema_ir=schema_ir,
    scope=scope,
)
if isinstance(adv_batch, BatchExtractionError):
    raise RuntimeError(f"Adversarial extraction failed: {adv_batch}")

m = adv_batch.metrics
print(f"\nPass 1: {m.total_proposal_count} proposals, {m.total_valid_count} valid, {m.total_rejection_count} rejections")
print(f"Gleaning: {adv_batch.gleaning_segments_reexamined} segments re-examined")
print(f"Total aggregated_specs: {len(adv_batch.aggregated_specs)}")

# Resolve
adv_resolved = entity_resolver.resolve_batch(list(adv_batch.aggregated_specs))
if isinstance(adv_resolved, ResolutionError):
    raise RuntimeError(f"Adversarial resolution failed: {adv_resolved}")
print(f"\nResolved specs: {len(adv_resolved.resolved_specs)}")
print(f"Merge events: {len(adv_resolved.merge_events)}")

In [ ]:
# ── Two-layer identity fragmentation diagnostic ──
from collections import Counter

def key_of(spec):
    return (spec.entity_type, tuple(sorted(spec.entity_identity.items())))

print("=== Layer 1: Extraction output (pre-resolver) ===")
raw_keys = [key_of(s) for s in adv_batch.aggregated_specs]
unique_raw = set(raw_keys)
print(f"unique entity keys: {len(unique_raw)}  |  total specs: {len(raw_keys)}")
for key, count in Counter(raw_keys).most_common():
    etype, identity = key
    print(f"  {etype:10s} {dict(identity)}: {count} spec(s)")

print(f"\n=== Layer 2: Resolver output (post-dedupe) ===")
resolved_keys = [key_of(s) for s in adv_resolved.resolved_specs]
unique_resolved = set(resolved_keys)
print(f"unique entity keys: {len(unique_resolved)}  |  total specs: {len(resolved_keys)}")
for key, count in Counter(resolved_keys).most_common():
    etype, identity = key
    print(f"  {etype:10s} {dict(identity)}: {count} spec(s)")

# ── Verdict ──
# Expected logical entities: ~4 (1 Document + 3 Modules: ingest, resolve, serve)
# If unique_resolved > 4, fragmentation candidates exist.
expected = 4
if len(unique_resolved) > expected:
    delta = len(unique_resolved) - expected
    print(f"\nFRAGMENTATION CANDIDATES: {len(unique_resolved)} resolved keys > {expected} expected")
    print(f"  {delta} excess key(s) — inspect identity dicts above for alias variants")
    print(f"  If human-verifiable as same entity -> P2 evidence (condition 1+2+3 met)")
else:
    print(f"\nNO FRAGMENTATION: {len(unique_resolved)} resolved keys <= {expected} expected")
    print(f"  Current resolver handled alias variants (or LLM normalized them).")
    print(f"  P2 not justified by this sample.")

### P0-off control experiment

Strip P0 entity context + P1 gleaning and re-run the same adversarial text.
If the raw LLM (no cross-segment context) produces different identity keys
for the same logical entity, that is genuine resolver-level fragmentation
evidence — the kind P2 would need to fix.

In [ ]:
# ── P0-off: raw LLM + basic resolver, no cross-segment context ──
from collections import Counter

raw_batch_extractor = BatchExtractor(
    extraction_agent=extraction_agent,
    batch_config=BatchExtractionConfig(
        max_segments_per_batch=1000,
        early_stop_on_batch_cap_reached=True,
        enable_entity_context=False,  # P0 OFF
        enable_gleaning=False,         # P1 OFF
    ),
)

raw_batch = raw_batch_extractor.extract_batch(
    segments=list(adv_staging.segments),
    schema_ir=schema_ir,
    scope=scope,
    source_doc_name="architecture_notes.md",
)
if isinstance(raw_batch, BatchExtractionError):
    raise RuntimeError(f"P0-off extraction failed: {raw_batch}")

m = raw_batch.metrics
print("=== P0-OFF extraction (no entity context, no gleaning) ===")
print(f"Pass 1: {m.total_proposal_count} proposals, {m.total_valid_count} valid, {m.total_rejection_count} rejections")
print(f"Gleaning: {raw_batch.gleaning_segments_reexamined} (expected 0)")
print(f"aggregated_specs: {len(raw_batch.aggregated_specs)}")

# Resolve
raw_resolved = entity_resolver.resolve_batch(list(raw_batch.aggregated_specs))
if isinstance(raw_resolved, ResolutionError):
    raise RuntimeError(f"P0-off resolution failed: {raw_resolved}")

def key_of(spec):
    return (spec.entity_type, tuple(sorted(spec.entity_identity.items())))

print(f"\n=== P0-OFF Layer 1: Extraction (pre-resolver) ===")
raw_keys = [key_of(s) for s in raw_batch.aggregated_specs]
unique_raw = set(raw_keys)
print(f"unique entity keys: {len(unique_raw)}  |  total specs: {len(raw_keys)}")
for key, count in Counter(raw_keys).most_common():
    etype, identity = key
    print(f"  {etype:10s} {dict(identity)}: {count} spec(s)")

print(f"\n=== P0-OFF Layer 2: Resolver (post-dedupe) ===")
resolved_keys = [key_of(s) for s in raw_resolved.resolved_specs]
unique_resolved = set(resolved_keys)
print(f"unique entity keys: {len(unique_resolved)}  |  total specs: {len(resolved_keys)}")
for key, count in Counter(resolved_keys).most_common():
    etype, identity = key
    print(f"  {etype:10s} {dict(identity)}: {count} spec(s)")

# ── Compare P0-on vs P0-off ──
print(f"\n=== COMPARISON ===")
print(f"P0-ON  unique keys (from earlier): check §10.5 diagnostic above")
print(f"P0-OFF unique keys: {len(unique_resolved)}")

expected = 4
if len(unique_resolved) > expected:
    delta = len(unique_resolved) - expected
    print(f"\nFRAGMENTATION DETECTED (P0-off): {len(unique_resolved)} keys > {expected} expected")
    print(f"  {delta} excess key(s) — P0 was masking resolver weakness")
    print(f"  P2 evidence condition 1+3 met. Check condition 2 (human-verifiable) above.")
else:
    print(f"\nNO FRAGMENTATION (even P0-off): {len(unique_resolved)} keys <= {expected} expected")
    print(f"  Raw LLM normalizes aliases without help. P2 genuinely not needed.")

## 11. Teardown

Close the SQLite-backed caches, close the runtime session, and remove the
temporary working directory. In a production agent this is the
try/finally block inside the runner's driver loop.

In [13]:
candidate_cache.close()
checkpoint_store.close()
close_resp = close_runtime_session(runtime_session_id)
print(f"runtime session closed: {close_resp.get('ok', False)}")

shutil.rmtree(workdir, ignore_errors=True)
print(f"workdir removed: {workdir}")

runtime session closed: True
workdir removed: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/notebook08_kgwg9vck


## 12. What this notebook demonstrated

One run of this notebook exercises the **entire agent layer contract** on a
real document:

| Layer | Archived blueprint | Exercised by |
|---|---|---|
| 4C1 — document staging | `2026-04-10_agent-layer4c1-document-staging` | `DocumentStaging.stage_document` |
| 4C3a — single-segment extract | `2026-04-10_agent-layer4c3a-single-segment-extraction` | `ExtractionAgent.extract_from_segment` |
| 4C3b — batch extract | `2026-04-10_agent-layer4c3b-batch-extraction` | `BatchExtractor.extract_batch` |
| 4C3c — entity resolution | `2026-04-10_agent-layer4c3c-entity-resolution` | `EntityResolver.resolve_batch` |
| 4C2 — draft bundle + commit | `2026-04-10_agent-layer4c2-draft-bundle-review` | `create_document_bundle` / `open_bundle_review` / `apply_bundle_review` / `commit_bundle` |
| Extraction prompt iter 5 | `2026-04-11_agent-extraction-prompt-semantic-grounding` (+ two earlier prompt fixes) | Every LLM call in cell 6 |
| Schema compilation | `kernel.sdk.compile_schema_from_classes` | Cell 2 — canonical-by-construction |
| Runtime session | `service.runtime_v1.open_runtime_session` | Cell 3 |

The B3 load test harness (`docs/references/working/load-test-2026-04-11/`) exercises
the same path against real PDF / DOCX / MD samples and persists per-run
records. Its `--commit` mode is currently blocked by an earlier approach that
hand-wrote a non-canonical schema fixture; see the two archived blueprints
`2026-04-11_load-test-runner-commit-path` and
`2026-04-11_b3-schema-canonicalization` for the history, and the carry-forward
note in `commit_path_plan_2026-04-11.md` for why future work on that line
should use `compile_schema_from_classes` (as this notebook does) instead of
probing the canonical validator rule by rule.

**OBS-01 variance** is expected: rerun cell 6 to see different proposal /
valid counts on the same input at `temperature=0.0`. `committed_count > 0` is
the only stable guarantee — the exact number is not.

---

**End of the public learning path.** Notebooks 01–07 cover the SDK and engine
layers; this notebook covers the agent layer end-to-end.